In [1]:
#Cell 1 : Imports & Paths
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

ROOT          = Path('..').resolve()
RAW_DIR       = ROOT / 'data' / 'raw' / 'ieee-fraud-detection'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_TXN = RAW_DIR / 'train_transaction.csv'
TRAIN_ID  = RAW_DIR / 'train_identity.csv'

assert TRAIN_TXN.exists(), f"Missing: {TRAIN_TXN}"
assert TRAIN_ID.exists(),  f"Missing: {TRAIN_ID}"
print("✅ Paths confirmed.")


✅ Paths confirmed.


In [2]:
#Cell 2 : Load & Merge Data
txn = pd.read_csv(TRAIN_TXN)
idn = pd.read_csv(TRAIN_ID)

df = pd.merge(txn, idn, how='left', on='TransactionID')

print(f"Transactions : {txn.shape[0]:,} rows × {txn.shape[1]} cols")
print(f"Identity     : {idn.shape[0]:,} rows × {idn.shape[1]} cols")
print(f"Merged       : {df.shape[0]:,} rows × {df.shape[1]} cols")

# Confirm no rows were lost in the left join
assert df.shape[0] == txn.shape[0], "Row count mismatch after merge — check for duplicate TransactionIDs"
print("✅ Merge verified — no rows lost.")


Transactions : 590,540 rows × 394 cols
Identity     : 144,233 rows × 41 cols
Merged       : 590,540 rows × 434 cols
✅ Merge verified — no rows lost.


In [3]:
#Cell 3 : Drop more than 95% columns
null_rates = df.isnull().mean()
cols_to_drop = null_rates[null_rates > 0.95].index.tolist()

print(f"Dropping {len(cols_to_drop)} columns with >95% nulls:")
print(cols_to_drop)

df.drop(columns=cols_to_drop, inplace=True)
print(f"\nShape after drop: {df.shape[0]:,} rows × {df.shape[1]} cols")


Dropping 9 columns with >95% nulls:
['id_07', 'id_08', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']

Shape after drop: 590,540 rows × 425 cols


In [4]:
#Cell 4 : Null-as-Signal flags
identity_signal_cols = ['id_30', 'id_31', 'id_33', 'id_36',
                        'DeviceType', 'DeviceInfo']

# Only flag columns that still exist after the drop step
identity_signal_cols = [c for c in identity_signal_cols if c in df.columns]

for col in identity_signal_cols:
    df[f'{col}_was_null'] = df[col].isnull().astype(int)

created = [f'{c}_was_null' for c in identity_signal_cols]
print(f"✅ Created {len(created)} null-signal flags: {created}")


✅ Created 6 null-signal flags: ['id_30_was_null', 'id_31_was_null', 'id_33_was_null', 'id_36_was_null', 'DeviceType_was_null', 'DeviceInfo_was_null']


/tmp/ipykernel_6075/3072810894.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_was_null'] = df[col].isnull().astype(int)
/tmp/ipykernel_6075/3072810894.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_was_null'] = df[col].isnull().astype(int)
/tmp/ipykernel_6075/3072810894.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a

In [5]:
#Cell 5 : Imputation
# Numerical → -999
num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
# Never impute the label
num_cols = [c for c in num_cols if c != 'isFraud']
df[num_cols] = df[num_cols].fillna(-999)

# Categorical → 'unknown'
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
df[cat_cols] = df[cat_cols].fillna('unknown')

remaining_nulls = df.isnull().sum().sum()
print(f"Remaining nulls after imputation: {remaining_nulls}")
assert remaining_nulls == 0, "Nulls remain — check dtypes"
print("✅ Imputation complete. Zero nulls remaining.")


/tmp/ipykernel_6075/3299324738.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns.tolist()


Remaining nulls after imputation: 0
✅ Imputation complete. Zero nulls remaining.


In [6]:
#Cell 6 : UID Reconstruction
# Primary UID: card + address + email domain
df['uid'] = (df['card1'].astype(str) + '_' +
             df['card2'].astype(str) + '_' +
             df['addr1'].astype(str) + '_' +
             df['P_emaildomain'].astype(str))

# Fallback UID: card + address only (for rows where email domain was null)
df['uid2'] = (df['card1'].astype(str) + '_' +
              df['card2'].astype(str) + '_' +
              df['addr1'].astype(str))

print(f"Unique uid  values: {df['uid'].nunique():,}")
print(f"Unique uid2 values: {df['uid2'].nunique():,}")
print(f"Total rows         : {df.shape[0]:,}")
print("✅ UID reconstruction complete.")


/tmp/ipykernel_6075/741291668.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['uid'] = (df['card1'].astype(str) + '_' +


Unique uid  values: 92,690
Unique uid2 values: 41,672
Total rows         : 590,540
✅ UID reconstruction complete.


/tmp/ipykernel_6075/741291668.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['uid2'] = (df['card1'].astype(str) + '_' +


In [7]:
#Cell 7 : Group Aggregation Features
agg_groups = ['card1', 'card2', 'P_emaildomain', 'uid', 'uid2', 'addr1']

for col in agg_groups:
    col_mean = df.groupby(col)['TransactionAmt'].transform('mean')
    col_std  = df.groupby(col)['TransactionAmt'].transform('std').fillna(0)
    col_count = df.groupby(col)['TransactionAmt'].transform('count')

    df[f'{col}_amt_mean']      = col_mean
    df[f'{col}_amt_std']       = col_std
    df[f'{col}_txn_count']     = col_count
    df[f'{col}_amt_deviation'] = df['TransactionAmt'] - col_mean
    df[f'{col}_amt_zscore']    = df[f'{col}_amt_deviation'] / (col_std + 1e-9)

new_agg_cols = len(agg_groups) * 5
print(f"✅ Created {new_agg_cols} group aggregation features.")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} cols")


/tmp/ipykernel_6075/275872887.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_amt_mean']      = col_mean
/tmp/ipykernel_6075/275872887.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_amt_std']       = col_std
/tmp/ipykernel_6075/275872887.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newfr

✅ Created 30 group aggregation features.
Shape: 590,540 rows × 463 cols


/tmp/ipykernel_6075/275872887.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_amt_mean']      = col_mean
/tmp/ipykernel_6075/275872887.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_amt_std']       = col_std
/tmp/ipykernel_6075/275872887.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newfr

In [8]:
#Cell 8 : Amount Decomposition
df['amt_integer']     = np.floor(df['TransactionAmt']).astype(int)
df['amt_decimal']     = df['TransactionAmt'] - df['amt_integer']
df['is_round_amount'] = (df['amt_decimal'] == 0).astype(int)
df['log_amt']         = np.log1p(df['TransactionAmt'])

print("✅ Amount decomposition complete.")
print(f"Round amount rate overall : {df['is_round_amount'].mean()*100:.2f}%")
print(f"Round amount rate — fraud : {df[df['isFraud']==1]['is_round_amount'].mean()*100:.2f}%")
print(f"Round amount rate — legit : {df[df['isFraud']==0]['is_round_amount'].mean()*100:.2f}%")


/tmp/ipykernel_6075/2899610872.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['amt_integer']     = np.floor(df['TransactionAmt']).astype(int)
/tmp/ipykernel_6075/2899610872.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['amt_decimal']     = df['TransactionAmt'] - df['amt_integer']
/tmp/ipykernel_6075/2899610872.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat

✅ Amount decomposition complete.
Round amount rate overall : 51.65%
Round amount rate — fraud : 52.66%
Round amount rate — legit : 51.61%


In [9]:
#Cell 9 : Time Features
df['hour']        = (df['TransactionDT'] // 3600) % 24
df['day_of_week'] = (df['TransactionDT'] // (3600 * 24)) % 7
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
df['is_night']    = ((df['hour'] < 6) | (df['hour'] >= 23)).astype(int)

print("✅ Time features created.")
print(df[['hour', 'day_of_week', 'is_weekend', 'is_night']].describe())


✅ Time features created.
             hour  day_of_week  is_weekend    is_night
count 590540.0000  590540.0000 590540.0000 590540.0000
mean      13.8619       2.9579      0.2882      0.3092
std        7.6072       2.0340      0.4529      0.4622
min        0.0000       0.0000      0.0000      0.0000
25%        6.0000       1.0000      0.0000      0.0000
50%       16.0000       3.0000      0.0000      0.0000
75%       20.0000       5.0000      1.0000      1.0000
max       23.0000       6.0000      1.0000      1.0000


/tmp/ipykernel_6075/3504074223.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['hour']        = (df['TransactionDT'] // 3600) % 24
/tmp/ipykernel_6075/3504074223.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['day_of_week'] = (df['TransactionDT'] // (3600 * 24)) % 7
/tmp/ipykernel_6075/3504074223.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead

In [10]:
#Cell 10 : Label Encoding
le = LabelEncoder()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove uid columns — they are used for groupby only, not fed to the model
uid_cols = ['uid', 'uid2', 'TransactionID']
cat_cols_to_encode = [c for c in cat_cols if c not in uid_cols]

for col in cat_cols_to_encode:
    df[col] = le.fit_transform(df[col].astype(str))

print(f"✅ Label encoded {len(cat_cols_to_encode)} categorical columns.")
print(f"Encoded: {cat_cols_to_encode}")


/tmp/ipykernel_6075/3002369417.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns.tolist()


✅ Label encoded 29 categorical columns.
Encoded: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [11]:
#Cell 11 : Drop non-feature columns
# These columns must not enter the model
drop_cols = ['TransactionID', 'uid', 'uid2']
drop_cols = [c for c in drop_cols if c in df.columns]

df.drop(columns=drop_cols, inplace=True)

print(f"✅ Dropped identifier columns: {drop_cols}")
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Feature count (excl. label): {df.shape[1] - 1}")


✅ Dropped identifier columns: ['TransactionID', 'uid', 'uid2']
Final shape: 590,540 rows × 468 cols
Feature count (excl. label): 467


In [12]:
#Cell 12 : Time-based train/val split
# Sort by TransactionDT to preserve temporal order
df_sorted = df.sort_values('TransactionDT').reset_index(drop=True)

split_idx  = int(len(df_sorted) * 0.8)
train_df   = df_sorted.iloc[:split_idx].copy()
val_df     = df_sorted.iloc[split_idx:].copy()

print(f"Train : {train_df.shape[0]:,} rows  |  Fraud: {train_df['isFraud'].sum():,}  ({train_df['isFraud'].mean()*100:.2f}%)")
print(f"Val   : {val_df.shape[0]:,} rows  |  Fraud: {val_df['isFraud'].sum():,}  ({val_df['isFraud'].mean()*100:.2f}%)")
print(f"\nTrain time range: {train_df['TransactionDT'].min()} → {train_df['TransactionDT'].max()}")
print(f"Val   time range: {val_df['TransactionDT'].min()} → {val_df['TransactionDT'].max()}")

# Confirm no temporal leakage — val must be strictly after train
assert val_df['TransactionDT'].min() >= train_df['TransactionDT'].max(), \
    "⚠️ Temporal leakage detected — val contains timestamps before end of train"
print("✅ No temporal leakage confirmed.")


Train : 472,432 rows  |  Fraud: 16,599  (3.51%)
Val   : 118,108 rows  |  Fraud: 4,064  (3.44%)

Train time range: 86400 → 12192842
Val   time range: 12192900 → 15811131
✅ No temporal leakage confirmed.


In [13]:
#Cell 13 : save processed data
train_path = PROCESSED_DIR / 'features_train.parquet'
val_path   = PROCESSED_DIR / 'features_val.parquet'

train_df.to_parquet(train_path, index=False)
val_df.to_parquet(val_path,   index=False)

print(f"✅ Saved: {train_path}")
print(f"✅ Saved: {val_path}")

# Reload sanity check
_check_train = pd.read_parquet(train_path)
_check_val   = pd.read_parquet(val_path)
print(f"\nReload check — train : {_check_train.shape}")
print(f"Reload check — val   : {_check_val.shape}")
assert 'isFraud' in _check_train.columns, "Label column missing from train parquet"
assert 'isFraud' in _check_val.columns,   "Label column missing from val parquet"
print("✅ Parquets verified. Ready for 03_lightgbm.ipynb.")


✅ Saved: /home/minipekka/Hackathons/USM VHack 2026/nuroguard/data/processed/features_train.parquet
✅ Saved: /home/minipekka/Hackathons/USM VHack 2026/nuroguard/data/processed/features_val.parquet

Reload check — train : (472432, 468)
Reload check — val   : (118108, 468)
✅ Parquets verified. Ready for 03_lightgbm.ipynb.
